# Beyond Prediction: Student Stress — Domain Ablation, SHAP ও Counterfactual Analysis

**Local Jupyter-এর জন্য | Baseline: Gradient Boosting | Dataset: `StressLevelDataset.csv` (1100 × 21)**

## এই notebook-এ আমরা কী করছি?
আগের paper-গুলো দেখিয়েছে *কোন model ভালো* এবং *কোন feature important*। এই notebook তিনটা নতুন প্রশ্নের উত্তর খোঁজে:

| RQ | প্রশ্ন | Experiment |
|---|---|---|
| RQ1 | কোন factor-domain prediction-এ সবচেয়ে গুরুত্বপূর্ণ? | Domain ablation + SHAP domain aggregation |
| RQ2 | একটা domain বাদ দিলে performance কতটা কমে? | Leave-one-domain-out ablation + Friedman/Wilcoxon |
| RQ3 | কোন individual feature class-wise সবচেয়ে বেশি প্রভাব ফেলে? | Global ও class-wise SHAP |
| RQ4 | কোন feature বদলালে model-এর predicted stress কমে? | Constrained counterfactual / what-if |
| RQ5 | Student profile অনুযায়ী counterfactual pattern আলাদা কি না? | Profile-wise counterfactual analysis |

## Novelty এক লাইনে
> আগের কাজ stress *predict ও explain* করেছে; আমরা দেখি **কোন factor-group ছাড়া prediction চলে না (necessity)** এবং **কী বদলালে model-এর prediction বদলায় (counterfactual)**।

## ⚠️ ভাষা ও claim-এর নিয়ম
Dataset **observational + cross-sectional + self-reported**। তাই:
- ✅ লিখবেন: *"Model অনুযায়ী sleep quality বাড়ালে predicted high-stress probability কমে।"*
- ❌ লিখবেন না: *"ঘুম ভালো হলে stress কমবে।"*
- Result আগে থেকে ধরে নেবেন না। "Psychological সবচেয়ে important" একটা **hypothesis**, conclusion নয়।

## Workflow
`Setup → Load/Audit → Domain mapping → Split → Baseline → Nested CV → Ablation → Stats → SHAP → Counterfactual → Profile analysis → Summary`

## ধাপ ১ — Setup
প্রয়োজনে প্রথমবার নিচের line-এর `#` তুলে চালান (তারপর kernel restart করুন):

In [ ]:
# [ধাপ ১] Library install (শুধু প্রথমবার; দরকার না হলে comment-ই থাকুক)
# %pip install shap scikit-learn pandas numpy matplotlib seaborn scipy

## ধাপ ২ — Import ও Global Config
সব গুরুত্বপূর্ণ setting এক জায়গায়। **Baseline freeze হওয়ার পর এগুলো বদলাবেন না** (বদলালে কারণ লিখে রাখুন)।
- `MAX_STEP`: counterfactual-এ একটা feature সর্বোচ্চ কত ধাপ বদলাতে পারবে
- `MAX_CHANGED`: একসাথে সর্বোচ্চ কয়টা feature বদলানো যাবে

In [ ]:
# [ধাপ ২] Import ও global config
import os, glob, json, itertools, warnings, platform
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    cross_validate, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import label_binarize

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print("shap install নেই -> SHAP ও profile analysis skip হবে। (%pip install shap)")

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ---------------- Config ----------------
DATA_PATH     = "StressLevelDataset.csv"   # CSV-টা notebook-এর পাশে রাখুন, নইলে path বদলান
TARGET        = "stress_level"
RANDOM_STATE  = 42
TEST_SIZE     = 0.20
N_SPLITS      = 5        # CV fold
N_REPEATS     = 3        # Repeated CV (ablation-এর statistical test-এর জন্য)
RUN_NESTED_CV = True     # ধীর মনে হলে False করুন
MAX_STEP      = 2        # counterfactual: প্রতি feature সর্বোচ্চ কত ধাপ বদলাবে
MAX_CHANGED   = 3        # counterfactual: একসাথে সর্বোচ্চ কয়টা feature
SHAP_BACKGROUND = 50     # SHAP background sample সংখ্যা (বাড়ালে ধীর কিন্তু স্থির)
SHAP_PERM       = 5      # SHAP permutation সংখ্যা (বাড়ালে ধীর কিন্তু কম noise)

os.makedirs("outputs", exist_ok=True)
print("Python", platform.python_version(), "| sklearn", sklearn.__version__,
      "| pandas", pd.__version__, "| shap", shap.__version__ if SHAP_OK else "N/A")

## ধাপ ৩ — Dataset Load ও Data Audit
**কেন?** Baseline-এর আগে নিশ্চিত হতে হবে: shape ঠিক আছে, missing/duplicate নেই, target-এর class balanced কি না।
প্রত্যাশা: **1100 row, 21 column, 3টা class (≈ 34/33/33%)**।

In [ ]:
# [ধাপ ৩] Dataset load ও audit
if not os.path.exists(DATA_PATH):
    found = sorted(glob.glob("*.csv") + glob.glob("data/*.csv"))
    raise FileNotFoundError(f"{DATA_PATH} পাওয়া যায়নি। কাছাকাছি CSV: {found}. DATA_PATH ঠিক করুন।")

df = pd.read_csv(DATA_PATH)
assert TARGET in df.columns, f"Target column '{TARGET}' নেই"

print("Shape            :", df.shape)
print("Total missing    :", int(df.isna().sum().sum()))
print("Duplicate rows   :", int(df.duplicated().sum()))
print("Classes          :", sorted(df[TARGET].unique()))

display(
    df[TARGET].value_counts().sort_index().to_frame("count")
      .assign(percent=lambda d: (d["count"] / len(df) * 100).round(2))
)
display(df.describe().T[["min", "max", "mean"]].round(2))

## ধাপ ৪ — Feature → Domain Mapping (Freeze)
২০টা predictor ৫টা domain-এ ভাগ (source paper-এর সাথে সামঞ্জস্যপূর্ণ, আপনার `workflow2_copy` থেকে)।
Mapping না মিললে code **error দেবে** — mapping guess করে বদলাবেন না।

**Modifiable vs Locked (counterfactual-এর জন্য):**
- **Modifiable** (hypothetically বদলানো যায়): `sleep_quality`, `social_support`, `study_load`, `extracurricular_activities`, `noise_level`, `living_conditions`
- **Locked** (বদলানো হবে না): `mental_health_history`, `blood_pressure`, `anxiety_level`, `depression`, `self_esteem`, ইত্যাদি বাকি সব
- `+1` মানে feature-এর মান *বাড়ানো* "ভালো" দিক; `-1` মানে *কমানো* ভালো দিক।

In [ ]:
# [ধাপ ৪] Domain mapping + modifiable/locked definition (freeze)
DOMAIN_FEATURES = {
    "psychological": ["anxiety_level", "self_esteem", "mental_health_history", "depression"],
    "physiological": ["headache", "blood_pressure", "sleep_quality", "breathing_problem"],
    "academic":      ["academic_performance", "study_load", "future_career_concerns"],
    "social":        ["teacher_student_relationship", "social_support",
                      "peer_pressure", "extracurricular_activities", "bullying"],
    "environmental": ["noise_level", "living_conditions", "safety", "basic_needs"],
}

# feature -> "ভালো" দিক (+1: বাড়ালে ভালো, -1: কমালে ভালো)
MODIFIABLE = {
    "sleep_quality": +1,
    "social_support": +1,
    "study_load": -1,
    "extracurricular_activities": +1,
    "noise_level": -1,
    "living_conditions": +1,
}

ALL_FEATURES = [c for c in df.columns if c != TARGET]
mapped = [f for fs in DOMAIN_FEATURES.values() for f in fs]

missing_in_data = sorted(set(mapped) - set(ALL_FEATURES))
unmapped        = sorted(set(ALL_FEATURES) - set(mapped))
dupe_mapped     = sorted({f for f in mapped if mapped.count(f) > 1})
bad_modifiable  = sorted(set(MODIFIABLE) - set(ALL_FEATURES))

print("Predictors in data :", len(ALL_FEATURES))
print("Mapped features    :", len(mapped))
print("Missing in data    :", missing_in_data)
print("Unmapped           :", unmapped)
print("Duplicate in map   :", dupe_mapped)
print("Bad modifiable     :", bad_modifiable)
if missing_in_data or unmapped or dupe_mapped or bad_modifiable:
    raise ValueError("Mapping মেলেনি — উপরের list দেখে ঠিক করুন; guess করে বদলাবেন না।")

FEATURE2DOMAIN = {f: d for d, fs in DOMAIN_FEATURES.items() for f in fs}
LOCKED = [f for f in ALL_FEATURES if f not in MODIFIABLE]
print("\nDomain size:", {d: len(fs) for d, fs in DOMAIN_FEATURES.items()})
print("Modifiable :", list(MODIFIABLE))
print("Locked     :", LOCKED)

## ধাপ ৫ — Train / Holdout Split (80:20, Stratified)
**Holdout (20%) পুরো গবেষণায় model select বা tune করতে ব্যবহার হবে না।** সব সিদ্ধান্ত train-এর CV থেকে; holdout শুধু final report ও explanation-এর জন্য।

In [ ]:
# [ধাপ ৫] Train/Holdout split
X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
CLASSES = sorted(y.unique())
HIGH = max(CLASSES)

print("Train:", X_train.shape, "| Holdout:", X_hold.shape)
display(pd.DataFrame({
    "Train %":   (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "Holdout %": (y_hold.value_counts(normalize=True).sort_index() * 100).round(2),
}))

### Sanity check — "ভালো দিক" ঠিক কি না
Counterfactual-এ আমরা ধরেছি (যেমন) `sleep_quality` বাড়ালে ভালো। **Data নিজে এটা সমর্থন করে কি?**
Train set-এ Spearman correlation: `feature × direction` এবং `stress_level`-এর মধ্যে সম্পর্ক **ঋণাত্মক** হওয়ার কথা (ভালো দিকে গেলে stress কম)।
উল্টো এলে (`⚠️`) ওই feature-এর direction পুনর্বিবেচনা করুন — dataset-এর encoding আপনার ধারণার সাথে নাও মিলতে পারে।

In [ ]:
# [ধাপ ৫] Modifiable feature-এর direction sanity check (শুধু train set)
rows = []
for f, d in MODIFIABLE.items():
    rho, p = stats.spearmanr(X_train[f] * d, y_train)
    rows.append({"feature": f, "direction": d, "spearman(feature*dir, stress)": round(rho, 3),
                 "check": "OK" if rho < 0 else "⚠️ direction উল্টো হতে পারে"})
display(pd.DataFrame(rows))

## ধাপ ৬ — Helper Function
- `make_pipeline()`: Imputer + Gradient Boosting। Imputer pipeline-এর ভেতরে থাকায় CV-তে **leakage হয় না**।
- `full_metrics()`: Accuracy, Balanced Acc, F1-weighted, F1-macro, MCC, ROC-AUC (OvR weighted), Brier।
- Brier = প্রতিটা sample-এ সব class-এর `(1 - p)²` যোগ, তারপর গড় (multiclass Brier; কম = ভালো)।

In [ ]:
# [ধাপ ৬] Helper function
def make_pipeline(**gb_params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE, **gb_params)),
    ])

def full_metrics(y_true, y_pred, y_prob, classes=CLASSES):
    yb = label_binarize(y_true, classes=classes)
    return {
        "Accuracy":          accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "F1-weighted":       f1_score(y_true, y_pred, average="weighted"),
        "F1-macro":          f1_score(y_true, y_pred, average="macro"),
        "MCC":               matthews_corrcoef(y_true, y_pred),
        "ROC-AUC (OvR)":     roc_auc_score(yb, y_prob, multi_class="ovr", average="weighted"),
        "Brier":             float(np.mean(np.sum((yb - y_prob) ** 2, axis=1))),
    }

def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(running, 1.0)
    return adj

print("Helper ready.")

## ধাপ ৭ — Baseline: Gradient Boosting (সব ২০ feature)
**কেন Gradient Boosting?** 2026 paper-এর predefined hierarchy-তে এটি best average ছিল (যদিও statistical পার্থক্য significant ছিল না)।
Default hyperparameter রাখছি, যাতে পরের ablation-এ পার্থক্য শুধু *feature বাদ দেওয়ার* কারণে আসে, tuning-এর কারণে নয়।

এই ধাপের ফল **freeze** হয়ে `outputs/baseline_results.json`-এ যাবে।

In [ ]:
# [ধাপ ৭] Baseline fit + holdout evaluation + freeze
baseline_model = make_pipeline().fit(X_train, y_train)
hold_pred = baseline_model.predict(X_hold)
hold_prob = baseline_model.predict_proba(X_hold)

baseline_hold = full_metrics(y_hold, hold_pred, hold_prob)
display(pd.DataFrame([baseline_hold], index=["Gradient Boosting (holdout)"]).round(4))

print(classification_report(y_hold, hold_pred, digits=4))

cm = confusion_matrix(y_hold, hold_pred, labels=CLASSES)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Baseline Confusion Matrix (Holdout)")
plt.tight_layout(); plt.show()

with open("outputs/baseline_results.json", "w") as f:
    json.dump({k: round(float(v), 6) for k, v in baseline_hold.items()}, f, indent=2)
print("Frozen -> outputs/baseline_results.json")

## ধাপ ৮ — Nested Cross-Validation (Tuning সহ unbiased estimate)
- **Outer 5-fold:** performance মাপে। **Inner 3-fold GridSearch:** hyperparameter বাছে।
- এটা বলে দেয় *tuning করলে* Gradient Boosting আসলে কত ভালো হবে — optimistic bias ছাড়া।
- Ablation (ধাপ ৯) default hyperparameter-এই চলবে; nested CV শুধু baseline-এর রিপোর্টের জন্য।

In [ ]:
# [ধাপ ৮] Nested CV (outer 5, inner 3)
if RUN_NESTED_CV:
    param_grid = {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [2, 3],
        "clf__learning_rate": [0.05, 0.1],
    }
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    search = GridSearchCV(make_pipeline(), param_grid, cv=inner, scoring="f1_weighted", n_jobs=-1)
    nested = cross_validate(
        search, X_train, y_train, cv=outer,
        scoring={"f1": "f1_weighted", "acc": "accuracy", "bal": "balanced_accuracy"},
        return_estimator=True, n_jobs=1
    )
    print(f"Nested CV F1-weighted : {nested['test_f1'].mean():.4f} ± {nested['test_f1'].std():.4f}")
    print(f"Nested CV Accuracy    : {nested['test_acc'].mean():.4f} ± {nested['test_acc'].std():.4f}")
    print(f"Nested CV Balanced Acc: {nested['test_bal'].mean():.4f} ± {nested['test_bal'].std():.4f}")
    print("\nপ্রতি outer fold-এ নির্বাচিত parameter:")
    for i, est in enumerate(nested["estimator"], 1):
        print(f"  fold {i}:", est.best_params_)
    nested_summary = {"F1-weighted mean": float(nested["test_f1"].mean()),
                      "F1-weighted sd": float(nested["test_f1"].std())}
else:
    nested_summary = None
    print("RUN_NESTED_CV=False -> skip")

## ধাপ ৯ — Domain Ablation (RQ1, RQ2) ⭐ প্রথম আসল নতুন experiment
**ধারণা:** SHAP বলে "কোন feature *important*"। Ablation বলে "কোন feature-group ছাড়া model *চলে না*" — এটাই **necessity**।

তিন ধরনের condition:
1. **All features** — reference (baseline)
2. **− domain** (leave-one-domain-out): একটা domain বাদ → performance কতটা কমে? (**বেশি কমলে ওই domain বেশি necessary**)
3. **only domain**: শুধু একটা domain রেখে → ওটা একা কতটা পারে? (sufficiency)

সব condition-এ **একই CV split** (`RepeatedStratifiedKFold`, 5×3 = 15 fold) এবং একই default Gradient Boosting — তাই fold-wise paired test করা যায়।
Holdout F1 শুধু তথ্য হিসেবে; সিদ্ধান্ত CV থেকে।

In [ ]:
# [ধাপ ৯] Domain ablation experiment
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

conditions = {"All features": ALL_FEATURES}
for d, fs in DOMAIN_FEATURES.items():
    conditions[f"- {d}"] = [f for f in ALL_FEATURES if f not in fs]
for d, fs in DOMAIN_FEATURES.items():
    conditions[f"only {d}"] = list(fs)

fold_f1, rows = {}, []
for name, cols in conditions.items():
    cv_out = cross_validate(
        make_pipeline(), X_train[cols], y_train, cv=rskf,
        scoring={"f1": "f1_weighted", "acc": "accuracy"}, n_jobs=-1
    )
    fold_f1[name] = cv_out["test_f1"]
    m = make_pipeline().fit(X_train[cols], y_train)
    hold_f1 = f1_score(y_hold, m.predict(X_hold[cols]), average="weighted")
    rows.append({
        "Condition": name, "n_features": len(cols),
        "CV F1 mean": cv_out["test_f1"].mean(), "CV F1 sd": cv_out["test_f1"].std(),
        "CV Acc mean": cv_out["test_acc"].mean(), "Holdout F1": hold_f1,
    })
    print(f"done: {name:28s} CV F1 = {cv_out['test_f1'].mean():.4f}")

ablation = pd.DataFrame(rows)
full_cv = ablation.loc[ablation["Condition"] == "All features", "CV F1 mean"].iloc[0]
full_hold = ablation.loc[ablation["Condition"] == "All features", "Holdout F1"].iloc[0]
ablation["ΔF1 (CV) vs All"] = ablation["CV F1 mean"] - full_cv
ablation["ΔF1 (Holdout) vs All"] = ablation["Holdout F1"] - full_hold

display(ablation.round(4))
ablation.round(5).to_csv("outputs/ablation_results.csv", index=False)

### Statistical test (ablation)
- **Friedman test:** All + ৫টা "− domain" condition-এর fold-wise F1-এ কোনো পার্থক্য আছে কি?
- **Wilcoxon signed-rank (paired):** প্রতিটা "− domain" বনাম "All features", **Holm correction** সহ।

⚠️ Repeated CV-র fold-গুলো সম্পূর্ণ independent নয় (training set overlap করে)। তাই p-value কিছুটা **optimistic** হতে পারে; paper-এ এই limitation লিখুন এবং p-value-র পাশাপাশি effect size (ΔF1) দেখান।

In [ ]:
# [ধাপ ৯] Statistical test: Friedman + Wilcoxon (Holm)
minus_names = ["All features"] + [f"- {d}" for d in DOMAIN_FEATURES]
fr = stats.friedmanchisquare(*[fold_f1[n] for n in minus_names])
print(f"Friedman test (All + 5 leave-one-domain-out): chi2 = {fr.statistic:.3f}, p = {fr.pvalue:.4g}")

tests = []
for n in minus_names[1:]:
    diff = fold_f1["All features"] - fold_f1[n]
    try:
        w = stats.wilcoxon(fold_f1["All features"], fold_f1[n])
        p = w.pvalue
    except ValueError:
        p = 1.0   # সব difference শূন্য
    tests.append({"Condition": n, "mean ΔF1 (All − cond)": diff.mean(), "raw p": p})

tests = pd.DataFrame(tests)
tests["Holm-adjusted p"] = holm_adjust(tests["raw p"].values)
tests["significant (α=0.05)"] = tests["Holm-adjusted p"] < 0.05
display(tests.round(4))
tests.round(5).to_csv("outputs/ablation_stats.csv", index=False)

In [ ]:
# [ধাপ ৯] Ablation plot
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

m = ablation[ablation["Condition"].str.startswith("- ")].copy()
m["drop"] = -m["ΔF1 (CV) vs All"]
m = m.sort_values("drop")
ax[0].barh(m["Condition"], m["drop"], color="tab:red")
ax[0].set_xlabel("F1 কমেছে (All − without domain)")
ax[0].set_title("Necessity: domain বাদ দিলে CV F1 কতটা কমে")

o = ablation[ablation["Condition"].str.startswith("only ")].sort_values("CV F1 mean")
ax[1].barh(o["Condition"], o["CV F1 mean"], xerr=o["CV F1 sd"], color="tab:blue")
ax[1].axvline(full_cv, color="k", ls="--", label="All features")
ax[1].set_xlabel("CV F1-weighted")
ax[1].set_title("Sufficiency: শুধু একটা domain দিয়ে")
ax[1].legend()
plt.tight_layout()
plt.savefig("outputs/ablation_plot.png", dpi=150)
plt.show()

## ধাপ ১০ — SHAP Analysis (RQ1, RQ3)
Baseline model (train-এ fit) এর SHAP, **holdout set-এ** হিসাব। Multiclass Gradient Boosting-এ `TreeExplainer` কাজ করে না, তাই model-agnostic **Permutation explainer** ব্যবহার হয়েছে; মান **probability space**-এ (অর্থাৎ SHAP = class-probability-তে অবদান)। Permutation SHAP approximate, তাই `SHAP_PERM` বাড়ালে ফল আরও স্থির হয় (ধীর হয়)।
- **Global/Class-wise:** কোন feature কোন class-এর prediction-এ কতটা প্রভাব ফেলে (`mean |SHAP|`)
- **Domain-wise:** feature-গুলোর `mean |SHAP|` domain অনুযায়ী যোগ করে % share

📌 SHAP একা novelty নয় (2026 paper আগেই করেছে)। এখানে SHAP-এর কাজ ablation ও counterfactual-কে **সমর্থন/তুলনা** করা: SHAP-এ top domain আর ablation-এ most-necessary domain কি এক?

In [ ]:
# [ধাপ ১০] SHAP (global, class-wise, domain-wise)
shap_vals = None
if SHAP_OK:
    # sklearn GradientBoosting (multiclass) shap.TreeExplainer সাপোর্ট করে না,
    # তাই model-agnostic Permutation explainer, predict_proba (probability space) এ।
    def predict_fn(arr):
        return baseline_model.predict_proba(pd.DataFrame(arr, columns=ALL_FEATURES))

    background = shap.sample(X_train, SHAP_BACKGROUND, random_state=RANDOM_STATE)
    explainer = shap.Explainer(
        predict_fn, background, algorithm="permutation",
        max_evals=2 * len(ALL_FEATURES) * SHAP_PERM + 1
    )
    expl = explainer(X_hold.values, silent=True)
    shap_vals = np.asarray(expl.values)      # shape: (n_samples, n_features, n_classes)
    print("SHAP shape:", shap_vals.shape)
    gb = baseline_model.named_steps["clf"]

    class_names = {c: f"class {c}" for c in CLASSES}
    imp_by_class = pd.DataFrame(
        np.abs(shap_vals).mean(axis=0), index=ALL_FEATURES,
        columns=[class_names[c] for c in gb.classes_]
    )
    imp_by_class["overall"] = imp_by_class.mean(axis=1)
    imp_by_class = imp_by_class.sort_values("overall", ascending=False)
    display(imp_by_class.head(10).round(4))

    top = imp_by_class.head(15).iloc[::-1]
    top[[class_names[c] for c in gb.classes_]].plot(kind="barh", stacked=True, figsize=(8, 6))
    plt.xlabel("mean |SHAP|"); plt.title("Top 15 Features (class-wise stacked)")
    plt.tight_layout(); plt.savefig("outputs/shap_features.png", dpi=150); plt.show()

    # Domain-wise aggregation
    dom = imp_by_class["overall"].groupby(imp_by_class.index.map(FEATURE2DOMAIN)).sum()
    dom_pct = (dom / dom.sum() * 100).sort_values(ascending=False)
    domain_shap = dom_pct.round(2).to_frame("SHAP share %")
    display(domain_shap)
    dom_pct.sort_values().plot(kind="barh", color="tab:green", figsize=(6, 3.5))
    plt.xlabel("SHAP share (%)"); plt.title("Domain-wise SHAP contribution")
    plt.tight_layout(); plt.savefig("outputs/shap_domains.png", dpi=150); plt.show()
    imp_by_class.round(5).to_csv("outputs/shap_feature_importance.csv")
else:
    print("shap নেই -> skip")

## ধাপ ১১ — Constrained Counterfactual / What-if Analysis (RQ4) ⭐ Main Contribution
**প্রশ্ন:** Model যে student-কে *High/Medium* বলছে, তার কয়টা **modifiable** feature সামান্য বদলালে model-এর predicted stress category নিচে নামে?

**Constraint (বাস্তবসম্মত রাখতে):**
1. শুধু `MODIFIABLE` feature বদলাবে; `LOCKED` সব অপরিবর্তিত।
2. শুধু **"ভালো দিকে"** বদলাবে (যেমন ঘুম বাড়বে, study load কমবে)।
3. প্রতি feature সর্বোচ্চ `MAX_STEP` ধাপ; মান train data-র observed range-এর ভেতরে।
4. একসাথে সর্বোচ্চ `MAX_CHANGED`টা feature।
5. **Minimal counterfactual:** সফল হলে *সবচেয়ে কম feature বদলে*, তারপর *সবচেয়ে কম মোট ধাপে* যেটা পারে সেটা নেওয়া হয়।

**ধাপ ১১ক:** একটা করে feature বদলালে (what-if) গড়ে P(High) কতটা বদলায়।
**ধাপ ১১খ:** Minimal counterfactual search।

⚠️ Limitation: feature-গুলো বাস্তবে পরস্পর সম্পর্কিত (ঘুম বদলালে উদ্বেগও বদলাতে পারে), কিন্তু এখানে বাকিগুলো স্থির ধরা হয়। এটা **model-based hypothetical**, বাস্তব intervention-এর ফল নয়।

In [ ]:
# [ধাপ ১১ক] Single-feature what-if (predicted High students)
model = baseline_model
CLF_CLASSES = list(model.named_steps["clf"].classes_)
IDX_HIGH = CLF_CLASSES.index(HIGH)
RANGES = {f: (int(X_train[f].min()), int(X_train[f].max())) for f in MODIFIABLE}

hold_pred_s = pd.Series(model.predict(X_hold), index=X_hold.index)
high_students = X_hold[hold_pred_s == HIGH]
print("Holdout-এ model যাদের High বলেছে:", len(high_students))

rows = []
for f, d in MODIFIABLE.items():
    lo, hi = RANGES[f]
    for s in range(1, MAX_STEP + 1):
        newv = high_students[f] + d * s
        ok = newv.between(lo, hi)
        if ok.sum() == 0:
            continue
        base = high_students[ok]
        mod = base.copy(); mod[f] = newv[ok]
        p0 = model.predict_proba(base)[:, IDX_HIGH]
        p1 = model.predict_proba(mod)[:, IDX_HIGH]
        rows.append({
            "feature": f, "step": s, "n": int(ok.sum()),
            "mean ΔP(High)": float((p1 - p0).mean()),
            "% predicted class নেমেছে": float((model.predict(mod) < HIGH).mean() * 100),
        })
whatif = pd.DataFrame(rows).sort_values(["step", "mean ΔP(High)"])
display(whatif.round(4))
whatif.round(5).to_csv("outputs/whatif_single_feature.csv", index=False)

piv = whatif.pivot(index="feature", columns="step", values="mean ΔP(High)")
piv.plot(kind="barh", figsize=(8, 4))
plt.axvline(0, color="k", lw=0.8)
plt.xlabel("গড় ΔP(High)  (ঋণাত্মক = predicted high-stress probability কমেছে)")
plt.title("Single-feature what-if (model-based)")
plt.tight_layout(); plt.savefig("outputs/whatif_single_feature.png", dpi=150); plt.show()

In [ ]:
# [ধাপ ১১খ] Minimal counterfactual search (constrained)
mod_features = list(MODIFIABLE)
col_idx = {c: i for i, c in enumerate(ALL_FEATURES)}

def generate_candidates(row):
    options = {}
    for f in mod_features:
        lo, hi = RANGES[f]; d = MODIFIABLE[f]
        vals = [row[f] + d * s for s in range(1, MAX_STEP + 1) if lo <= row[f] + d * s <= hi]
        if vals:
            options[f] = vals
    avail = list(options)
    cands = []
    for k in range(1, MAX_CHANGED + 1):
        for subset in itertools.combinations(avail, k):
            for vals in itertools.product(*[options[f] for f in subset]):
                cands.append(dict(zip(subset, vals)))
    return cands

def minimal_counterfactual(x_row, orig_class):
    cands = generate_candidates(x_row)
    if not cands:
        return None
    arr = np.tile(x_row.values.astype(float), (len(cands), 1))
    for i, ch in enumerate(cands):
        for f, v in ch.items():
            arr[i, col_idx[f]] = v
    cdf = pd.DataFrame(arr, columns=ALL_FEATURES)
    proba = model.predict_proba(cdf)
    new_cls = np.array(CLF_CLASSES)[proba.argmax(axis=1)]
    idx_orig = CLF_CLASSES.index(orig_class)
    ok = np.where(new_cls < orig_class)[0]
    if len(ok) == 0:
        return None
    # minimal: কম feature -> কম মোট ধাপ -> orig class-এর probability সবচেয়ে কম
    def key(i):
        ch = cands[i]
        steps = sum(abs(v - x_row[f]) for f, v in ch.items())
        return (len(ch), steps, proba[i, idx_orig])
    best = min(ok, key=key)
    return {"changes": cands[best], "n_changed": len(cands[best]),
            "new_class": int(new_cls[best]),
            "p_before": float(model.predict_proba(x_row.to_frame().T.astype(float))[0, idx_orig]),
            "p_after": float(proba[best, idx_orig])}

hold_pred_arr = hold_pred_s.values
cf_rows = []
for pos, (idx, x_row) in enumerate(X_hold.iterrows()):
    oc = int(hold_pred_arr[pos])
    if oc == min(CLASSES):
        continue                      # Low-stress student-এর জন্য নামার জায়গা নেই
    res = minimal_counterfactual(x_row, oc)
    rec = {"pos": pos, "index": idx, "orig_class": oc, "found": res is not None}
    if res:
        rec.update(res)
    cf_rows.append(rec)

cf = pd.DataFrame(cf_rows)
print("Counterfactual search হয়েছে:", len(cf), "জন (predicted Medium/High)")
summ = (cf.groupby("orig_class")
          .agg(students=("found", "size"), found=("found", "sum"))
          .assign(success_pct=lambda d: (d["found"] / d["students"] * 100).round(1)))
display(summ)

found = cf[cf["found"]]
if len(found):
    print("\nসফল counterfactual-এ বদলানো feature-সংখ্যা:")
    display(found["n_changed"].value_counts().sort_index().to_frame("students"))
    freq = Counter(f for ch in found["changes"] for f in ch)
    freq_s = pd.Series(freq).sort_values(ascending=False)
    display(freq_s.to_frame("কতবার minimal counterfactual-এ এসেছে"))
    freq_s.iloc[::-1].plot(kind="barh", figsize=(7, 3.5), color="tab:purple")
    plt.title("কোন feature বেশি বদলাতে হয়েছে (minimal counterfactual)")
    plt.tight_layout(); plt.savefig("outputs/cf_feature_frequency.png", dpi=150); plt.show()

    print("\nকয়েকটা উদাহরণ (model-based hypothetical):")
    for _, r in found.head(3).iterrows():
        orig = X_hold.loc[r["index"]]
        print(f"\nStudent index {r['index']}: class {int(r['orig_class'])} -> {int(r['new_class'])}")
        for f, v in r["changes"].items():
            print(f"   {f}: {orig[f]:.0f} -> {v:.0f}")
        print(f"   P(original class): {r['p_before']:.2f} -> {r['p_after']:.2f}")
cf.drop(columns=["changes"], errors="ignore").to_csv("outputs/counterfactual_results.csv", index=False)

## ধাপ ১২ — Profile-wise Analysis (RQ5)
**প্রশ্ন:** সব High-stress student-এর জন্য কি একই পরিবর্তন কাজ করে, নাকি profile অনুযায়ী আলাদা?

**Profile কীভাবে ঠিক হয়:** প্রতিটা predicted-High student-এর জন্য SHAP থেকে দেখা হয় কোন **domain** তাকে "High" দিকে সবচেয়ে বেশি ঠেলছে। ওই domain-ই তার *dominant profile*।
তারপর profile-ভিত্তিক তুলনা: counterfactual সফলতার হার, কয়টা feature বদলাতে হয়, কোন feature বেশি আসে।

⚠️ কোনো profile-এ ৫ জনের কম student থাকলে সেটা দুর্বল প্রমাণ — সংখ্যা দেখে সাবধানে পড়ুন।

In [ ]:
# [ধাপ ১২] Profile-wise counterfactual analysis (SHAP-dominant domain অনুযায়ী)
if SHAP_OK and shap_vals is not None:
    feat_pos = {d: [ALL_FEATURES.index(f) for f in fs] for d, fs in DOMAIN_FEATURES.items()}
    high_cols = [i for i, c in enumerate(model.named_steps["clf"].classes_) if c == HIGH][0]

    prof_rows = []
    for pos in np.where(hold_pred_arr == HIGH)[0]:
        contrib = {d: shap_vals[pos, ix, high_cols].sum() for d, ix in feat_pos.items()}
        prof_rows.append({"pos": int(pos), "profile": max(contrib, key=contrib.get)})
    prof = pd.DataFrame(prof_rows)

    cf_high = cf[cf["orig_class"] == HIGH].merge(prof, on="pos", how="inner")
    out = []
    for p, g in cf_high.groupby("profile"):
        gf = g[g["found"]]
        freq = Counter(f for ch in gf["changes"] for f in ch)
        out.append({
            "profile (dominant domain)": p,
            "n students": len(g),
            "CF success %": round(len(gf) / len(g) * 100, 1),
            "mean #features changed": round(gf["n_changed"].mean(), 2) if len(gf) else np.nan,
            "mean P(High) drop": round((gf["p_before"] - gf["p_after"]).mean(), 3) if len(gf) else np.nan,
            "top changed features": ", ".join(f"{f}({c})" for f, c in freq.most_common(3)),
            "⚠️ n<5": len(g) < 5,
        })
    profile_table = pd.DataFrame(out).sort_values("n students", ascending=False)
    display(profile_table)
    profile_table.to_csv("outputs/profile_analysis.csv", index=False)
else:
    profile_table = None
    print("shap নেই -> profile analysis skip")

## ধাপ ১৩ — Final Summary ও RQ-wise উত্তর
নিচের cell তথ্য সংগ্রহ করে একটা summary দেয়। **এটা শুধু সংখ্যা সাজায়; ব্যাখ্যা ও conclusion আপনি data দেখে লিখবেন।**

In [ ]:
# [ধাপ ১৩] Summary
print("=" * 70)
print("RESEARCH SUMMARY (সব সংখ্যা এই dataset ও এই split-এর জন্য)")
print("=" * 70)

print(f"\n[Baseline] Gradient Boosting holdout: Acc={baseline_hold['Accuracy']:.4f}, "
      f"F1w={baseline_hold['F1-weighted']:.4f}, MCC={baseline_hold['MCC']:.4f}")
if nested_summary:
    print(f"[Nested CV] F1-weighted = {nested_summary['F1-weighted mean']:.4f} ± {nested_summary['F1-weighted sd']:.4f}")

m = ablation[ablation["Condition"].str.startswith("- ")].copy()
top_nec = m.sort_values("ΔF1 (CV) vs All").iloc[0]
print(f"\n[RQ2 Ablation] সবচেয়ে বেশি কমে যায় যে condition: {top_nec['Condition']} "
      f"(ΔF1 = {top_nec['ΔF1 (CV) vs All']:+.4f})")
print(f"   Friedman p = {fr.pvalue:.4g}")
print("   Holm-adjusted significant:", tests.loc[tests['significant (α=0.05)'], 'Condition'].tolist() or "কোনোটাই নয়")

if SHAP_OK and shap_vals is not None:
    print(f"\n[RQ1 SHAP] সবচেয়ে বেশি SHAP share-এর domain: {dom_pct.index[0]} ({dom_pct.iloc[0]:.1f}%)")
    print("   Top-5 feature:", imp_by_class.index[:5].tolist())

if len(cf):
    print(f"\n[RQ4 Counterfactual] predicted Medium/High student-দের {cf['found'].mean()*100:.1f}% "
          f"-এর জন্য ≤{MAX_CHANGED}টা modifiable feature (≤{MAX_STEP} ধাপ) বদলে নিচের category পাওয়া গেছে।")
    best1 = whatif[whatif['step'] == 1].sort_values('mean ΔP(High)').head(1)
    if len(best1):
        r = best1.iloc[0]
        print(f"   Single-feature (step=1) সবচেয়ে বড় গড় P(High) কমা: {r['feature']} ({r['mean ΔP(High)']:+.3f})")

summary = {
    "baseline_holdout": {k: round(float(v), 5) for k, v in baseline_hold.items()},
    "nested_cv": nested_summary,
    "ablation_largest_drop": top_nec["Condition"],
    "friedman_p": float(fr.pvalue),
    "cf_success_rate_pct": round(float(cf["found"].mean() * 100), 2) if len(cf) else None,
}
with open("outputs/summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("\nসব output 'outputs/' folder-এ সেভ হয়েছে।")

## Paper-এ লেখার নিয়ম (Checklist)

**১. যা লিখবেন (উদাহরণ)**
- *"Removing the **X** domain produced the largest decrease in CV F1-weighted (ΔF1 = …), suggesting it is the most necessary domain for prediction in this dataset."*
- *"Under the model, increasing sleep quality by one level was associated with a mean decrease of … in predicted high-stress probability."*
- *"For …% of students predicted as high-stress, changing at most three modifiable features altered the model's predicted category."*

**২. যা লিখবেন না**
- ❌ "ঘুম/support বাড়ালে stress কমবে" (causal claim)
- ❌ "Psychological domain সবচেয়ে important" — যদি ablation/SHAP তা না দেখায়
- ❌ "Gradient Boosting সবচেয়ে ভালো" — শুধু একটা CV score দেখে (2026 paper-ও significant পার্থক্য পায়নি)

**৩. Limitation অংশে অবশ্যই থাকবে**
1. Observational, cross-sectional, self-reported data → causal নয়।
2. একটাই dataset (1100 student); external validation নেই → সাধারণীকরণ সীমিত।
3. Repeated-CV fold independent নয় → p-value কিছুটা optimistic।
4. Counterfactual-এ বাকি feature স্থির ধরা হয়েছে, বাস্তবে feature-গুলো correlated।
5. Modifiable/locked তালিকা ও "ভালো দিক" গবেষকের সিদ্ধান্ত।

**৪. RQ → সংশ্লিষ্ট অংশ**
| RQ | কোন ধাপের output |
|---|---|
| RQ1 | ধাপ ৯ (only-domain) + ধাপ ১০ (domain SHAP) |
| RQ2 | ধাপ ৯ (leave-one-out + test) |
| RQ3 | ধাপ ১০ (class-wise SHAP) |
| RQ4 | ধাপ ১১ক, ১১খ |
| RQ5 | ধাপ ১২ |

**৫. পরের ধাপ (ঐচ্ছিক)** — আরও শক্ত করতে: Random Forest/XGBoost দিয়ে একই ablation (robustness), অন্য random seed-এ পুনরাবৃত্তি, MAX_STEP/MAX_CHANGED পরিবর্তন করে sensitivity analysis।